In [1]:
# ============================================================
# SRI LANKA TOURISM - EXPLORATORY DATA ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD CLEAN DATA
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

input_file = PROCESSED_DIR / "tourism_arrivals_clean.csv"

tourism = pd.read_csv(input_file)

tourism["Date"] = pd.to_datetime(tourism["Date"])

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Rows:", len(tourism))
print("Countries:", tourism["Country_Standardized"].nunique())
print("Years:", sorted(tourism["Year"].unique()))
print("Date range:", tourism["Date"].min().date(), "to", tourism["Date"].max().date())


# ------------------------------------------------------------
# 2. BASIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC SUMMARY")
print("=" * 70)

display(tourism.describe(include="all"))


# ------------------------------------------------------------
# 3. TOTAL ARRIVALS BY YEAR
# ------------------------------------------------------------

yearly_arrivals = (
    tourism.groupby("Year", as_index=False)["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)

print("\n" + "=" * 70)
print("TOTAL ARRIVALS BY YEAR")
print("=" * 70)

display(yearly_arrivals)


# ------------------------------------------------------------
# 4. YEAR-OVER-YEAR GROWTH
# ------------------------------------------------------------

yearly_arrivals["YoY_Growth_%"] = (
    yearly_arrivals["Arrivals"]
    .pct_change() * 100
)

print("\n" + "=" * 70)
print("YEAR-OVER-YEAR GROWTH")
print("=" * 70)

display(yearly_arrivals)


# ------------------------------------------------------------
# 5. MONTHLY TOTAL ARRIVALS
# ------------------------------------------------------------

monthly_arrivals = (
    tourism.groupby(
        ["Year", "Month_Number", "Month"],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
    .sort_values(["Year", "Month_Number"])
)

print("\n" + "=" * 70)
print("MONTHLY ARRIVALS")
print("=" * 70)

display(monthly_arrivals)


# ------------------------------------------------------------
# 6. BEST AND WORST MONTHS
# ------------------------------------------------------------

best_months = (
    monthly_arrivals
    .sort_values("Arrivals", ascending=False)
    .head(10)
)

worst_months = (
    monthly_arrivals
    .sort_values("Arrivals", ascending=True)
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 MONTHS BY ARRIVALS")
print("=" * 70)

display(best_months)

print("\n" + "=" * 70)
print("LOWEST 10 MONTHS BY ARRIVALS")
print("=" * 70)

display(worst_months)


# ------------------------------------------------------------
# 7. TOP 10 SOURCE COUNTRIES
# ------------------------------------------------------------

country_arrivals = (
    tourism.groupby("Country_Standardized", as_index=False)["Arrivals"]
    .sum(min_count=1)
    .sort_values("Arrivals", ascending=False)
)

print("\n" + "=" * 70)
print("TOP 10 SOURCE COUNTRIES")
print("=" * 70)

display(country_arrivals.head(10))


# ------------------------------------------------------------
# 8. TOP 20 SOURCE COUNTRIES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 SOURCE COUNTRIES")
print("=" * 70)

display(country_arrivals.head(20))


# ------------------------------------------------------------
# 9. COUNTRY SHARE OF TOTAL ARRIVALS
# ------------------------------------------------------------

total_arrivals = tourism["Arrivals"].sum()

country_arrivals["Share_%"] = (
    country_arrivals["Arrivals"] / total_arrivals * 100
)

print("\n" + "=" * 70)
print("TOP COUNTRIES BY SHARE OF TOTAL ARRIVALS")
print("=" * 70)

display(country_arrivals.head(15))


# ------------------------------------------------------------
# 10. AVERAGE MONTHLY ARRIVALS BY COUNTRY
# ------------------------------------------------------------

country_monthly_avg = (
    tourism.groupby("Country_Standardized", as_index=False)["Arrivals"]
    .mean()
    .sort_values("Arrivals", ascending=False)
)

country_monthly_avg.columns = [
    "Country",
    "Average_Monthly_Arrivals"
]

print("\n" + "=" * 70)
print("TOP COUNTRIES BY AVERAGE MONTHLY ARRIVALS")
print("=" * 70)

display(country_monthly_avg.head(15))


# ------------------------------------------------------------
# 11. SEASONALITY
# ------------------------------------------------------------

seasonality = (
    tourism.groupby(
        ["Month_Number", "Month"],
        as_index=False
    )["Arrivals"]
    .mean()
    .sort_values("Month_Number")
)

print("\n" + "=" * 70)
print("AVERAGE ARRIVALS BY MONTH - SEASONALITY")
print("=" * 70)

display(seasonality)


# ------------------------------------------------------------
# 12. 2026 JAN-OCT VS PREVIOUS YEARS JAN-OCT
# ------------------------------------------------------------

ytd_comparison = (
    tourism[
        tourism["Month_Number"] <= 10
    ]
    .groupby("Year", as_index=False)["Arrivals"]
    .sum(min_count=1)
    .sort_values("Year")
)

ytd_comparison["YoY_Growth_%"] = (
    ytd_comparison["Arrivals"]
    .pct_change() * 100
)

print("\n" + "=" * 70)
print("JANUARY-OCTOBER COMPARISON")
print("=" * 70)

display(ytd_comparison)


# ------------------------------------------------------------
# 13. COUNTRY RANKING BY YEAR
# ------------------------------------------------------------

country_year = (
    tourism.groupby(
        ["Year", "Country_Standardized"],
        as_index=False
    )["Arrivals"]
    .sum(min_count=1)
)

country_year["Rank"] = (
    country_year.groupby("Year")["Arrivals"]
    .rank(
        method="dense",
        ascending=False
    )
)

country_year = country_year.sort_values(
    ["Year", "Rank"]
)

print("\n" + "=" * 70)
print("TOP 10 COUNTRIES BY YEAR")
print("=" * 70)

display(
    country_year[
        country_year["Rank"] <= 10
    ]
)


# ------------------------------------------------------------
# 14. DATA QUALITY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EDA DATA QUALITY SUMMARY")
print("=" * 70)

print("Missing arrivals:", tourism["Arrivals"].isna().sum())
print("Negative arrivals:", (tourism["Arrivals"] < 0).sum())
print(
    "Duplicate country-month records:",
    tourism.duplicated(
        subset=[
            "Country_Standardized",
            "Year",
            "Month_Number"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 15. SAVE EDA TABLES
# ------------------------------------------------------------

eda_dir = PROJECT_ROOT / "data" / "processed" / "eda"

eda_dir.mkdir(
    parents=True,
    exist_ok=True
)

yearly_arrivals.to_csv(
    eda_dir / "yearly_arrivals.csv",
    index=False
)

monthly_arrivals.to_csv(
    eda_dir / "monthly_arrivals.csv",
    index=False
)

country_arrivals.to_csv(
    eda_dir / "country_arrivals.csv",
    index=False
)

seasonality.to_csv(
    eda_dir / "seasonality.csv",
    index=False
)

ytd_comparison.to_csv(
    eda_dir / "ytd_comparison.csv",
    index=False
)

country_year.to_csv(
    eda_dir / "country_year_ranking.csv",
    index=False
)

print("\n" + "=" * 70)
print("EDA COMPLETE")
print("=" * 70)

print("EDA tables saved to:")
print(eda_dir)

DATASET LOADED
Rows: 11110
Countries: 229
Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Date range: 2022-01-01 to 2026-10-01

BASIC SUMMARY


,Country_Raw,Country_Standardized,Year,Month_Number,Month,Date,Arrivals
count,11110,11110,11110.000000,11110.000000,11110,11110,11108.000000
unique,229,229,NaN,NaN,12,NaN,NaN
top,Afghanistan,Afghanistan,NaN,NaN,January,NaN,NaN
freq,58,58,NaN,NaN,958,NaN,NaN
mean,NaN,NaN,2023.948875,6.326283,NaN,2024-05-23 02:22:34.455445,763.915286
min,NaN,NaN,2022.000000,1.000000,NaN,2022-01-01 00:00:00,0.000000
25%,NaN,NaN,2023.000000,3.000000,NaN,2023-03-01 00:00:00,2.000000
50%,NaN,NaN,2024.000000,6.000000,NaN,2024-06-01 00:00:00,15.000000
75%,NaN,NaN,2025.000000,9.000000,NaN,2025-08-01 00:00:00,213.250000
max,NaN,NaN,2026.000000,12.000000,NaN,2026-10-01 00:00:00,62211.000000



TOTAL ARRIVALS BY YEAR


,Year,Arrivals
0,2022,719978.0
1,2023,1487303.0
2,2024,2053465.0
3,2025,2362521.0
4,2026,1862304.0



YEAR-OVER-YEAR GROWTH


,Year,Arrivals,YoY_Growth_%
0,2022,719978.0,NaN
1,2023,1487303.0,106.576173
2,2024,2053465.0,38.066352
3,2025,2362521.0,15.050463
4,2026,1862304.0,-21.173018


... (17 outputs left out)

,Year,Arrivals,YoY_Growth_%
0,2022,568258.0,NaN
1,2023,1125455.0,98.053525
2,2024,1620715.0,44.005313
3,2025,1890687.0,16.657586
4,2026,1862304.0,-1.501200



TOP 10 COUNTRIES BY YEAR


,Year,Country_Standardized,Arrivals,Rank
74,2022,India,123004.0,1.0
138,2022,Russian Federation,91272.0,2.0
177,2022,United Kingdom,85187.0,3.0
61,2022,Germany,55542.0,4.0
57,2022,France,35482.0,5.0
9,2022,Australia,30924.0,6.0
30,2022,Canada,26845.0,7.0
178,2022,United States,22230.0,8.0
102,2022,Maldives,18880.0,9.0
133,2022,Poland,15195.0,10.0



EDA DATA QUALITY SUMMARY
Missing arrivals: 2
Negative arrivals: 0
Duplicate country-month records: 0

EDA COMPLETE
EDA tables saved to:
c:\DataAnalyticsPortfolio\01_Projects\01_Sri_Lanka_Tourism_Analytics\data\processed\eda
